# TLINK Binary Classification - Pythia-70M + Class Weights

**Task:** Given a sentence with two event/time spans marked by XML tags, predict:
- `YES` - a temporal link exists between them
- `NO` - no temporal relationship exists

**Model:** `EleutherAI/pythia-70m` (70M parameters)

**Key improvement:** Class weights to handle the 91%/9% YES/NO imbalance.



## Cell 1 - Check GPU


In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU - go to Runtime -> Change runtime type -> T4 GPU')


GPU: Tesla T4
Memory: 15.6 GB


## Cell 2 - Install packages

**After this cell finishes: Runtime -> Restart session, then run from Cell 3 onward.**


In [ ]:
!pip uninstall -q -y transformers tokenizers accelerate
!pip install -q transformers datasets accelerate scikit-learn
print('Done')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.1 MB/s eta 0:00:00
Done. Now: Runtime -> Restart session, then run from Cell 3.


## Cell 3 - Imports

Run this after restarting the session.


In [ ]:
import os, time
import torch
import numpy as np
from torch import nn
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report,
)
import transformers
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
print(f'GPU          : {torch.cuda.is_available()}')
print('All imports OK')


transformers : 5.6.2
torch        : 2.10.0+cu128
GPU          : True
All imports OK


## Cell 4 - Config


In [ ]:
MODEL_NAME    = 'EleutherAI/pythia-70m'
DATASET_NAME  = 'mdg-nlp/tlink-extr-classification-sentence-2-label'
OUTPUT_DIR    = './tlink_model'
MAX_LENGTH    = 256
BATCH_SIZE    = 32
LEARNING_RATE = 2e-6
NUM_EPOCHS    = 5
EARLY_STOPPING_PATIENCE = 3
LABEL2ID = {'NO': 0, 'YES': 1}
ID2LABEL = {0: 'NO', 1: 'YES'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}')


Device: cuda, LR: 2e-06, Batch: 32


## Cell 5 - Load dataset

The dataset has a large class imbalance: ~91% YES, ~9% NO.
A naive model that always predicts YES would score 91% accuracy but 0% recall on NO.
We fix this with class weights in Cell 10.


In [ ]:
dataset = load_dataset(DATASET_NAME)
print(f"Train     : {len(dataset['train']):,}")
print(f"Validation: {len(dataset['validation']):,}")
print(f"Test      : {len(dataset['test']):,}")
counts = Counter(dataset['train']['label'])
total  = len(dataset['train'])
print('\nLabel distribution:')
for lbl, cnt in sorted(counts.items()):
    print(f'  {lbl}: {cnt:,} ({cnt/total*100:.1f}%)')
print('\nSample examples:')
for i in range(2):
    ex = dataset['train'][i]
    print(f"  [{ex['label']}] {ex['text'][:100]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train     : 61,440
Validation: 7,422
Test      : 200

Label distribution:
  NO: 5,500 (9.0%)
  YES: 55,940 (91.0%)

Sample examples:
  [YES] AFP_ENG_20061205.0069

MANILA, Dec 5 , 2006, 2006

US demands custody of marine convicted in Philipp
  [YES] AFP_ENG_20061205.0069

MANILA, <t1>Dec 5 , 2006</t1>, 2006

US demands custody of marine convicted i


## Cell 6 - Tokenizer

Three Pythia-specific fixes:
1. **No pad token** - reuse EOS token as pad
2. **Special tags** - register `<e1>`, `<e2>` etc. as single tokens
3. **Left-padding** - Pythia classifies from the last token position


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
special_tokens = ['<e1>', '</e1>', '<e2>', '</e2>', '<t1>', '</t1>', '<t2>', '</t2>']
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
tokenizer.padding_side = 'left'
print(f'Vocab size   : {len(tokenizer):,}')
print(f'Pad token    : {tokenizer.pad_token}')
print(f'Padding side : {tokenizer.padding_side}')


Vocab size   : 50,285
Pad token    : <|endoftext|>
Padding side : left


## Cell 7 - Tokenize dataset


In [ ]:
def tokenize(batch):
    encoding = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    encoding['labels'] = [LABEL2ID[lbl] for lbl in batch['label']]
    return encoding

tokenized = dataset.map(tokenize, batched=True, batch_size=1000)
tokenized = tokenized.remove_columns(['id', 'doc_id', 'text', 'label'])
tokenized.set_format('torch')
print(f'Columns    : {tokenized["train"].column_names}')
print(f'Train size : {len(tokenized["train"]):,}')


Map:   0%|          | 0/7422 [00:00<?, ? examples/s]

Columns    : ['input_ids', 'attention_mask', 'labels']
Train size : 61,440


## Cell 8 - Load model

Adds a classification head: last token hidden state (512-dim) -> Linear(512->2)




In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))
total = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total:,}')
print(f'Device     : {DEVICE}')


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-70m
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters : 44,662,272
Device     : cuda


## Cell 9 - Metrics

We track both binary F1 (YES class) and macro F1 (average across both classes).
We use **macro F1** as the checkpoint metric because it rewards the model equally
for getting both YES and NO right, not just the majority class.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'  : accuracy_score(labels, preds),
        'f1_yes'    : f1_score(labels, preds, average='binary', pos_label=1),
        'f1_no'     : f1_score(labels, preds, average='binary', pos_label=0),
        'f1_macro'  : f1_score(labels, preds, average='macro'),
        'precision' : precision_score(labels, preds, average='binary', pos_label=1, zero_division=0),
        'recall'    : recall_score(labels, preds, average='binary', pos_label=1, zero_division=0),
    }
print('compute_metrics defined')


compute_metrics defined


## Cell 10 - Class weights and WeightedTrainer

### Why class weights?
The default cross-entropy loss treats every example equally.
With 91% YES examples, the model learns to just predict YES for everything.

Class weights make the loss penalise mistakes on the minority class (NO) more heavily.
We compute weights as: `total / (num_classes * class_count)`
- NO  weight = 61440 / (2 * 5500)  = **5.58** (minority, penalised more)
- YES weight = 61440 / (2 * 55940) = **0.55** (majority, penalised less)

### How it works in code
HuggingFace Trainer does not natively support class weights, so we subclass it
and override only the `compute_loss` method. Everything else (evaluation,
checkpointing, early stopping) stays exactly the same.


In [ ]:
# Compute class weights from training label counts
no_count  = 5500
yes_count = 55940
total_count = no_count + yes_count
num_classes = 2

import math
w_no  = math.sqrt(total_count / (num_classes * no_count))
w_yes = math.sqrt(total_count / (num_classes * yes_count))

# [NO weight, YES weight] - index matches LABEL2ID: NO=0, YES=1
class_weights = torch.tensor([w_no, w_yes]).to(device=DEVICE, dtype=model.dtype)
print(f'Class weights: NO={w_no:.2f}, YES={w_yes:.2f}')

class WeightedTrainer(Trainer):
    """
    Subclass of Trainer that replaces the default cross-entropy loss
    with a class-weighted version to handle label imbalance.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=False,
    bf16=False,
    dataloader_num_workers=2,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
print('WeightedTrainer configured')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Class weights: NO=2.36, YES=0.74
WeightedTrainer configured


## Cell 11 - Train

Expected: ~5-10 min per epoch on T4 GPU.
Watch that `f1_no` increases from epoch 1 - that is the sign class weights are working.


In [ ]:
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'Training complete in {elapsed/60:.1f} minutes')


Epoch,Training Loss,Validation Loss,Accuracy,F1 Yes,F1 No,F1 Macro,Precision,Recall
1,0.677452,0.468994,0.888844,0.939352,0.335214,0.637283,0.929985,0.948908
2,0.453318,0.470459,0.920641,0.957914,0.306243,0.632078,0.923024,0.995544
3,0.456918,0.443115,0.909458,0.951283,0.360000,0.655642,0.929189,0.974454
4,0.414048,0.433838,0.913366,0.953497,0.367748,0.660623,0.929236,0.979058
5,0.363564,0.427246,0.910132,0.951537,0.382979,0.667258,0.931437,0.972523


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete in 16.6 minutes


## Cell 12 - Evaluate

The classification report breaks down precision, recall, and F1 **per class**.
With class weights working correctly, the NO row should now have meaningful scores.


In [ ]:
print('=== Validation set ===')
val_results = trainer.evaluate(tokenized['validation'])
for k, v in val_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print('\n=== Test set - full classification report ===')
test_out    = trainer.predict(tokenized['test'])
preds       = np.argmax(test_out.predictions, axis=-1)
true_labels = test_out.label_ids
print(classification_report(true_labels, preds, target_names=['NO', 'YES']))


=== Validation set ===


Training Loss,Validation Loss,Epoch,Accuracy,F1 Yes,F1 No,F1 Macro,Precision,Recall
0.363564,0.427246,5,0.910132,0.951537,0.382979,0.667258,0.931437,0.972523


  eval_loss: 0.4272
  eval_accuracy: 0.9101
  eval_f1_yes: 0.9515
  eval_f1_no: 0.3830
  eval_f1_macro: 0.6673
  eval_precision: 0.9314
  eval_recall: 0.9725

=== Test set - full classification report ===


              precision    recall  f1-score   support

          NO       0.97      0.30      0.46       100
         YES       0.59      0.99      0.74       100

    accuracy                           0.65       200
   macro avg       0.78      0.65      0.60       200
weighted avg       0.78      0.65      0.60       200



## Cell 13 - Save model


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}  ({size/1e6:.1f} MB)')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./tlink_model/
  checkpoint-1920  (0.0 MB)
  checkpoint-3840  (0.0 MB)
  checkpoint-5760  (0.0 MB)
  checkpoint-7680  (0.0 MB)
  checkpoint-9600  (0.0 MB)
  config.json  (0.0 MB)
  model.safetensors  (89.3 MB)
  tokenizer.json  (3.6 MB)
  tokenizer_config.json  (0.0 MB)
  training_args.bin  (0.0 MB)


## Cell 14 - Download model to your computer

This zips the model and downloads it directly. More reliable than Google Drive mounting.


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('tlink_model', 'zip', '.', 'tlink_model')
files.download('tlink_model.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 15 - Inference demo

Run these examples to see how the model performs after class weight training.


In [ ]:
from transformers import pipeline
clf = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)
examples = [
    ('The department <e1>said</e1> <t1>Wednesday</t1> the US filed a note.', 'YES'),
    ('The marine was <e1>sentenced</e1> for <e2>raping</e2> a woman.',        'NO'),
    ('She <e1>arrived</e1> at the office <t1>before noon</t1>.',              'YES'),
    ('The vote was <e1>counted</e1> <t1>after</t1> polls <e2>closed</e2>.',  'YES'),
]
print(f"{'Text':<58} {'Expected':<10} {'Got':<6} {'Conf'}")
print('-' * 82)
correct = 0
for text, expected in examples:
    r = clf(text, truncation=True, max_length=MAX_LENGTH)[0]
    mark = 'OK' if r['label'] == expected else 'WRONG'
    correct += int(r['label'] == expected)
    print(f"{text[:56]:<58} {expected:<10} {r['label']:<6} {r['score']:.3f}  {mark}")
print(f'\nDemo accuracy: {correct}/{len(examples)}')


Text                                                       Expected   Got    Conf
----------------------------------------------------------------------------------
The department <e1>said</e1> <t1>Wednesday</t1> the US f   YES        YES    0.905  OK
The marine was <e1>sentenced</e1> for <e2>raping</e2> a    NO         YES    0.816  WRONG
She <e1>arrived</e1> at the office <t1>before noon</t1>.   YES        YES    0.549  OK
The vote was <e1>counted</e1> <t1>after</t1> polls <e2>c   YES        YES    0.912  OK

Demo accuracy: 3/4
